# Lung Disease V2: multi-label chest X-ray classification on NIH ChestX-ray14

V1 (MSc project) treated every label combination as its own class (836 classes), reported accuracy only,
and split images at random, so the same patient could appear in training and validation.

V2 fixes all three:
- **14-label multi-label task** with sigmoid outputs and binary cross-entropy (with class weights).
- **Official patient-wise split** (`train_val_list.txt` / `test_list.txt`), and a validation set split by patient.
- **Per-class AUROC** on the official test set, with a bootstrap 95% interval for the mean.

Models: DenseNet121 and ResNet50, ImageNet-pretrained, same pipeline and settings.

**Settings before running:** Accelerator = GPU (T4 or P100), Internet = On (to download ImageNet weights).
Add the dataset `nih-chest-xrays/data` with "Add Input". Then "Save Version" and choose "Save & Run All".

Not a medical device. Results are for research only and must not be used for diagnosis.

In [ ]:
from __future__ import annotations

import glob
import json
import os
import time
from multiprocessing import Pool
from pathlib import Path

import numpy as np
import pandas as pd

SMOKE = False          # True = tiny run to check the pipeline (a few minutes)
IMG = 224
BATCH = 128
MAX_EPOCHS = 8
PATIENCE = 2
LR = 1e-4
SEED = 42
VAL_FRACTION = 0.1
TIME_BUDGET_PER_MODEL_H = 4.5
MODELS = ["densenet121", "resnet50"]
LABELS = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass", "Nodule", "Pneumonia",
    "Pneumothorax", "Consolidation", "Edema", "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia",
]
OUT = Path(os.environ.get("OUT_DIR", "/kaggle/working/results"))
CACHE = Path(os.environ.get("CACHE_DIR", "/tmp/cxr_cache"))
INPUT_ROOT = os.environ.get("INPUT_ROOT", "/kaggle/input")
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

## 1. Find the data and build the patient-wise split

In [ ]:
def find_one(name: str) -> Path:
    hits = glob.glob(os.path.join(INPUT_ROOT, "**", name), recursive=True)
    if not hits:
        raise FileNotFoundError(name + " not found under " + INPUT_ROOT + ". Add the nih-chest-xrays/data dataset.")
    return Path(hits[0])


def index_images(root: str) -> dict:
    paths = {}
    for dirpath, _, files in os.walk(root):
        for f in files:
            if f.endswith(".png"):
                paths[f] = os.path.join(dirpath, f)
    return paths


def multi_hot(finding_labels: pd.Series) -> np.ndarray:
    y = np.zeros((len(finding_labels), len(LABELS)), dtype=np.float32)
    pos = {name: i for i, name in enumerate(LABELS)}
    for row, text in enumerate(finding_labels):
        for name in str(text).split("|"):
            if name in pos:
                y[row, pos[name]] = 1.0
    return y


def patient_split(df: pd.DataFrame, fraction: float, seed: int) -> tuple:
    patients = np.array(sorted(df["Patient ID"].unique()))
    rng = np.random.default_rng(seed)
    rng.shuffle(patients)
    n_val = max(1, int(round(len(patients) * fraction)))
    val_patients = set(patients[:n_val].tolist())
    is_val = df["Patient ID"].isin(val_patients).to_numpy()
    return df[~is_val].reset_index(drop=True), df[is_val].reset_index(drop=True)


def load_splits() -> tuple:
    entry = find_one("Data_Entry_2017.csv")
    root = entry.parent
    df = pd.read_csv(entry)
    images = index_images(str(root))
    df = df[df["Image Index"].isin(images)].copy()
    df["path"] = df["Image Index"].map(images)
    train_val_names = set(find_one("train_val_list.txt").read_text().split())
    test_names = set(find_one("test_list.txt").read_text().split())
    train_val = df[df["Image Index"].isin(train_val_names)].reset_index(drop=True)
    test = df[df["Image Index"].isin(test_names)].reset_index(drop=True)
    train, val = patient_split(train_val, VAL_FRACTION, SEED)
    if SMOKE:
        train, val, test = train.head(2000), val.head(500), test.head(500)
    shared = set(train["Patient ID"]) & (set(val["Patient ID"]) | set(test["Patient ID"]))
    assert not (set(train["Patient ID"]) & set(val["Patient ID"])), "patient leakage train/val"
    assert not (set(train_val["Patient ID"]) & set(test["Patient ID"])), "patient leakage train_val/test"
    print("images found:", len(df), "| train", len(train), "val", len(val), "test", len(test),
          "| patients shared train vs val/test:", len(shared))
    return train, val, test

## 2. Resize every image once to 224x224 (grayscale) and cache it
Decoding 1024x1024 PNGs every epoch is the slowest part, so we do it once.

In [ ]:
def _load_small(path: str) -> np.ndarray:
    from PIL import Image
    with Image.open(path) as im:
        return np.asarray(im.convert("L").resize((IMG, IMG), Image.BILINEAR), dtype=np.uint8)


def build_cache(df: pd.DataFrame, name: str) -> Path:
    file = CACHE / (name + "_" + str(IMG) + "_" + str(len(df)) + ".npy")
    if file.exists():
        return file
    arr = np.lib.format.open_memmap(str(file), mode="w+", dtype=np.uint8, shape=(len(df), IMG, IMG))
    t0 = time.time()
    with Pool(os.cpu_count() or 2) as pool:
        for i, img in enumerate(pool.imap(_load_small, df["path"].tolist(), chunksize=64)):
            arr[i] = img
            if i % 10000 == 0:
                print(name, i, "/", len(df), round(time.time() - t0), "s")
    arr.flush()
    del arr
    return file

## 3. Dataset, models, training and evaluation

In [ ]:
def run_all() -> None:
    import torch
    import torch.nn as nn
    from sklearn.metrics import roc_auc_score
    from torch.utils.data import DataLoader, Dataset
    from torchvision import models

    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

    train_df, val_df, test_df = load_splits()
    caches = {n: build_cache(d, n) for n, d in [("train", train_df), ("val", val_df), ("test", test_df)]}
    targets = {"train": multi_hot(train_df["Finding Labels"]), "val": multi_hot(val_df["Finding Labels"]),
               "test": multi_hot(test_df["Finding Labels"])}

    class CachedXray(Dataset):
        def __init__(self, file: Path, y: np.ndarray):
            self.file, self.y, self.arr = str(file), y, None

        def __len__(self) -> int:
            return len(self.y)

        def __getitem__(self, i: int):
            if self.arr is None:
                self.arr = np.load(self.file, mmap_mode="r")
            return torch.from_numpy(np.array(self.arr[i])), torch.from_numpy(self.y[i])

    workers = min(4, os.cpu_count() or 2)
    loaders = {
        split: DataLoader(CachedXray(caches[split], targets[split]), batch_size=BATCH, shuffle=(split == "train"),
                          num_workers=workers, pin_memory=True, persistent_workers=workers > 0)
        for split in ["train", "val", "test"]
    }
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

    def prepare(x: torch.Tensor, train: bool) -> torch.Tensor:
        x = x.to(device, non_blocking=True).float().div_(255.0).unsqueeze(1).repeat(1, 3, 1, 1)
        if train:
            flip = torch.rand(x.shape[0], device=device) < 0.5
            x[flip] = x[flip].flip(-1)
        return (x - mean) / std

    def build(name: str) -> nn.Module:
        if name == "densenet121":
            m = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
            m.classifier = nn.Linear(m.classifier.in_features, len(LABELS))
        elif name == "resnet50":
            m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            m.fc = nn.Linear(m.fc.in_features, len(LABELS))
        else:
            raise ValueError(name)
        return m.to(device)

    @torch.no_grad()
    def predict(model: nn.Module, loader) -> np.ndarray:
        model.eval()
        out = []
        for x, _ in loader:
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                out.append(torch.sigmoid(model(prepare(x, False))).float().cpu().numpy())
        return np.concatenate(out)

    def per_class_auroc(y: np.ndarray, p: np.ndarray) -> list:
        scores = []
        for k in range(len(LABELS)):
            scores.append(float(roc_auc_score(y[:, k], p[:, k])) if 0 < y[:, k].sum() < len(y) else float("nan"))
        return scores

    def bootstrap_mean_auroc(y: np.ndarray, p: np.ndarray, n: int = 200) -> tuple:
        rng = np.random.default_rng(SEED)
        means = []
        for _ in range(n):
            idx = rng.integers(0, len(y), len(y))
            means.append(np.nanmean(per_class_auroc(y[idx], p[idx])))
        return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

    pos = targets["train"].sum(0)
    pos_weight = torch.tensor(np.clip((len(targets["train"]) - pos) / np.maximum(pos, 1), 1, 50),
                              dtype=torch.float32, device=device)
    summary = {}
    per_class_rows = []
    for name in MODELS:
        model = build(name)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=0)
        scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        best, bad, history, t0 = -1.0, 0, [], time.time()
        weights_file = OUT / (name + "_best.pt")
        for epoch in range(1 if SMOKE else MAX_EPOCHS):
            model.train()
            total, seen = 0.0, 0
            for x, y in loaders["train"]:
                y = y.to(device, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                    loss = loss_fn(model(prepare(x, True)), y)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
                total += loss.item() * len(y)
                seen += len(y)
            val_auc = float(np.nanmean(per_class_auroc(targets["val"], predict(model, loaders["val"]))))
            sched.step(val_auc)
            hours = (time.time() - t0) / 3600
            history.append({"model": name, "epoch": epoch + 1, "train_loss": total / seen, "val_mean_auroc": val_auc,
                            "hours": round(hours, 3)})
            print(history[-1])
            if val_auc > best:
                best, bad = val_auc, 0
                torch.save(model.state_dict(), weights_file)
            else:
                bad += 1
            if bad >= PATIENCE or hours > TIME_BUDGET_PER_MODEL_H:
                break
        model.load_state_dict(torch.load(weights_file, map_location=device))
        probs = predict(model, loaders["test"])
        scores = per_class_auroc(targets["test"], probs)
        low, high = bootstrap_mean_auroc(targets["test"], probs)
        summary[name] = {"test_mean_auroc": float(np.nanmean(scores)), "ci95": [low, high],
                         "best_val_mean_auroc": best, "epochs_run": len(history),
                         "train_hours": history[-1]["hours"]}
        per_class_rows += [{"model": name, "label": lab, "test_auroc": s} for lab, s in zip(LABELS, scores)]
        pd.DataFrame(history).to_csv(OUT / (name + "_history.csv"), index=False)
        np.save(OUT / (name + "_test_probs.npy"), probs.astype(np.float16))
        print(name, summary[name])
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()

    run_info = {"smoke": SMOKE, "img": IMG, "batch": BATCH, "lr": LR, "seed": SEED, "torch": torch.__version__,
                "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
                "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
                "n_patients_train": int(train_df["Patient ID"].nunique())}
    (OUT / "summary.json").write_text(json.dumps({"run": run_info, "models": summary}, indent=2))
    pd.DataFrame(per_class_rows).to_csv(OUT / "per_class_auroc.csv", index=False)
    np.save(OUT / "test_targets.npy", targets["test"].astype(np.uint8))
    print(json.dumps(summary, indent=2))

## 4. Run (about 3 to 6 hours on a Kaggle T4/P100 for both models)

In [ ]:
run_all()

## 5. Charts

In [ ]:
def make_charts() -> None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    table = pd.read_csv(OUT / "per_class_auroc.csv").pivot(index="label", columns="model", values="test_auroc")
    table = table.loc[LABELS]
    ax = table.plot.barh(figsize=(8, 7), color=["#1f4e79", "#c9a227"][: len(table.columns)])
    ax.axvline(0.5, color="grey", linestyle=":", linewidth=1)
    ax.set_xlim(0.4, 1.0)
    ax.set_xlabel("Test AUROC (official patient-wise test split)")
    ax.set_title("Per-class AUROC, NIH ChestX-ray14")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(OUT / "per_class_auroc.png", dpi=150)
    plt.close()

    fig, ax = plt.subplots(figsize=(7, 4))
    for name in MODELS:
        f = OUT / (name + "_history.csv")
        if f.exists():
            h = pd.read_csv(f)
            ax.plot(h["epoch"], h["val_mean_auroc"], marker="o", label=name)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation mean AUROC")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT / "val_auroc_curves.png", dpi=150)
    plt.close()
    print(table.round(3))


make_charts()